In [1]:
from PIL import Image
import requests
import torch
from transformers import CLIPProcessor, CLIPModel
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

model.to(device)
model.eval()


dummy_text = "a photo of a cat"

/home/phli/genAI/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/phli/genAI/.venv/lib64/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
# Load images based on plaintext_data.txt
data_dir = "/home/phli/genAI/cp4101_data"
text_file = os.path.join(data_dir, "plaintext_data.txt")
image_dir = os.path.join(data_dir, "images")

grouped_images = []

with open(text_file, "r") as f:
    for line in f:
        line = line.strip()
        
        # Start a new image group if a new base image is found
        if line.startswith("base"):
            line = line[4:]
            grouped_images.append([])
        
        # load the image
        image_filename = line.split(">")[0]
        grouped_images[-1].append(Image.open(os.path.join(image_dir, image_filename)))

In [3]:
res = 0
num_image_groups = len(grouped_images)
for i, group in enumerate(grouped_images):
    print(f"Image group {i}: {len(group)}")
    res += len(group)
print(f"Total {res} of {num_image_groups} groups")

Image group 0: 40
Image group 1: 130
Image group 2: 158
Image group 3: 198
Image group 4: 238
Image group 5: 60
Image group 6: 552
Total 1376 of 7 groups


In [ ]:
combined_image_embeds = []
for group in grouped_images[:1]:
    torch.no_grad()
    inputs = processor(text=dummy_text, images=group[:10], return_tensors="pt", padding=True)
    inputs.to(device)
    raw_embeds = model(**inputs).image_embeds
    print(raw_embeds.shape)

torch.Size([10, 768])
